In [ ]:
from pathlib import Path
import os
import gc
from contextlib import contextmanager

def absolute(path):
    return Path(os.path.abspath(os.fspath(path)))

current = absolute(Path.cwd())
ROOT = current.parent if current.name == 'src' else current
WORK_DIR = ROOT.parent / 'Code_run'

OUTPUT_DIR = ROOT / 'output'
EVALUATION_BRANCHES = ['county_holdout', 'utility_holdout']

In [ ]:
import numpy as np
import pandas as pd

FEATURES = ['ELEP_M', 'LST_HDD', 'LST_CDD', 'LST_STD', 'NTL', 'PM25', 'NDVI', 'NDBI', 'EVI', 'LONGITUDE', 'LATITUDE', 'POP_DENS', 'PCT_DEM', 'AC_RECS', 'ACS_MEDIAN_HH_INC', 'ACS_PCT_HU_ELEC', 'ACS_PCT_HU_SOLAR', 'ACS_PCT_OWNER_HU_COST_30PCT', 'ACS_PCT_RENTER_HU', 'ACS_MEDIAN_RENT', 'ACS_MEDIAN_HOME_VALUE', 'ACS_PCT_COMMT_60MINUP', 'ACS_PCT_BLACK_NONHISP', 'ACS_PCT_HISPANIC']
LABEL = 'ELEC_M'
SEASONS = ['spring', 'summer', 'fall', 'winter']
SEASON_MONTHS = {'spring': [3, 4, 5], 'summer': [6, 7, 8], 'fall': [9, 10, 11], 'winter': [1, 2, 12]}
FEATURE_LABELS = {'LST_HDD': 'Heating degree days', 'LST_CDD': 'Cooling degree days', 'LST_STD': 'Land surface temperature variability', 'NTL': 'Nighttime lights intensity', 'PM25': 'PM2.5', 'NDVI': 'Normalized difference vegetation index', 'NDBI': 'Normalized difference built-up index', 'EVI': 'Enhanced vegetation index', 'LONGITUDE': 'Longitude', 'LATITUDE': 'Latitude', 'ELEP_M': 'Monthly electricity price', 'POP_DENS': 'Population density', 'PCT_DEM': '% of democrats', 'AC_RECS': '% of AC adoption', 'ACS_MEDIAN_HH_INC': 'Median household income', 'ACS_PCT_HU_ELEC': '% of housing units heating with electricity', 'ACS_PCT_HU_SOLAR': '% of housing units heating with solar', 'ACS_PCT_OWNER_HU_COST_30PCT': '% of owners with housing costs >= 30% of income', 'ACS_PCT_RENTER_HU': '% of renters', 'ACS_MEDIAN_RENT': 'Median rent', 'ACS_MEDIAN_HOME_VALUE': 'Median home value', 'ACS_PCT_COMMT_60MINUP': '% of commute >= 60 minutes', 'ACS_PCT_BLACK_NONHISP': '% of African American (non-Hispanic)', 'ACS_PCT_HISPANIC': '% of Hispanic'}
SEED = 3407

FI_REPEATS = 5
SHAP_BACKGROUND_ROWS = 100
SHAP_BACKGROUND_SEED = 42
SHAP_EXPLAINED_ROWS = 500
SHAP_EXPLAINED_SEED = 43
SHAP_NSAMPLES = 1000
SHAP_VERSION = '0.50.0'
SHAP_L1_REG = 'num_features(10)'

In [ ]:
def training_frame(season, branch='county_holdout'):
    frame = pd.read_parquet(ROOT / 'data/training' / f'{season}.parquet').copy()
    if branch == 'utility_holdout':
        frame['partition'] = frame['utility_partition']
        frame['_county_fold'] = frame['utility_fold']
        frame['_utility_group'] = frame['utility_group']
    return frame

def model_metrics(actual, predicted):
    from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
    actual, predicted = np.asarray(actual, float), np.asarray(predicted, float)
    return {
        'n': len(actual), 'r2': float(r2_score(actual, predicted)),
        'rmse': float(mean_squared_error(actual, predicted) ** 0.5),
        'mape_percent': float(np.mean(np.abs(predicted-actual)/actual)*100),
        'mae': float(mean_absolute_error(actual, predicted)),
        'bias': float(np.mean(predicted-actual)),
    }

@contextmanager
def model_for_evaluation(model_path):
    from autogluon.tabular import TabularPredictor
    predictor = TabularPredictor.load(str(model_path), verbosity=0)
    try:
        yield predictor
    finally:
        try:
            predictor.unpersist(models='all')
        finally:
            del predictor
            gc.collect()

In [ ]:
def samples(frame,features):
    columns=['GEOID','MONTH',*features,'ELEC_M']
    full=frame[columns].copy()
    pool=full.loc[np.isfinite(full[features].to_numpy(float)).all(axis=1)].copy()
    background=pool.sample(n=SHAP_BACKGROUND_ROWS,random_state=SHAP_BACKGROUND_SEED).reset_index(drop=True)
    reference=pool.sample(n=SHAP_EXPLAINED_ROWS,random_state=SHAP_EXPLAINED_SEED).reset_index(drop=True)
    return background,reference,reference.copy()

def common_test(county,utility):
    keys=['GEOID','MONTH']
    a=county.loc[county.partition.eq('test')].copy()
    b=utility.loc[utility.partition.eq('test'),keys]
    result=a.merge(b,on=keys,how='inner',validate='one_to_one').sort_values(keys).reset_index(drop=True)
    return result

def permuted_features(frame,features,feature_index,repeat,seed=3407):
    x=frame[features];shuffled=x.copy()
    rng=np.random.RandomState(seed+feature_index*101+repeat)
    for month in sorted(frame.MONTH.unique()):
        positions=np.flatnonzero(frame.MONTH.to_numpy()==month)
        shuffled.iloc[positions,feature_index]=rng.permutation(x.iloc[positions,feature_index].to_numpy())
    return shuffled

def importance_table(records):
    from scipy.stats import t
    table=pd.DataFrame(records).groupby('feature',sort=False).importance.agg(['mean','std','count']).reset_index()
    table=table.rename(columns={'mean':'importance','std':'stddev','count':'n'})
    ci=t.ppf(.995,table.n-1)*table.stddev/np.sqrt(table.n)
    table['p99_low']=table.importance-ci;table['p99_high']=table.importance+ci
    return table

def compute_importance(frame, features, predict):
    target = frame.ELEC_M.to_numpy(float)
    baseline = float(np.sqrt(np.mean((predict(frame[features]) - target) ** 2)))
    records = []
    for feature_index, feature in enumerate(features):
        for repeat in range(FI_REPEATS):
            changed = predict(permuted_features(frame, features, feature_index, repeat))
            rmse = float(np.sqrt(np.mean((changed - target) ** 2)))
            records.append({'feature': feature, 'repeat': repeat,
                            'importance': rmse - baseline})
    return importance_table(records)

def compute_shap(frame, features, predict):
    import shap
    background, reference, explained = samples(frame, features)
    predictions = predict(reference[features])
    explainer = shap.KernelExplainer(predict, background[features], link='identity')
    baseline = float(explainer.expected_value)
    values = []
    for i in range(SHAP_EXPLAINED_ROWS):
        np.random.seed(SEED + i)
        value = np.asarray(explainer.shap_values(
            explained[features].iloc[i:i+1], nsamples=SHAP_NSAMPLES,
            l1_reg=SHAP_L1_REG, silent=True)).reshape(-1)
        assert value.shape == (len(features),) and np.isfinite(value).all()
        assert abs(baseline + value.sum() - predictions[i]) < 0.05
        values.append(value)
    return pd.DataFrame(np.asarray(values), columns=features), explained

In [ ]:
def calculate_explanations(season):
    county = training_frame(season, 'county_holdout')
    utility = training_frame(season, 'utility_holdout')
    with model_for_evaluation(WORK_DIR / 'models' / 'county_holdout' / season) as predictor:
        predictor.persist(models='best', with_ancestors=True, max_memory=.5)

        def predict(values):
            frame = values if isinstance(values, pd.DataFrame) else pd.DataFrame(values, columns=FEATURES)
            result = predictor.predict(frame[FEATURES]).to_numpy(dtype=np.float64)
            return result

        importance = compute_importance(common_test(county, utility), FEATURES, predict)
        values, explained = compute_shap(county, FEATURES, predict)

    WORK_DIR.mkdir(parents=True, exist_ok=True)
    importance.to_csv(WORK_DIR / f'county_holdout_{season}_feature_importance.csv', index=False)
    values.to_parquet(WORK_DIR / f'county_holdout_{season}_shap_values.parquet', index=False)
    explained.to_parquet(WORK_DIR / f'county_holdout_{season}_explained_500.parquet', index=False)

for season in SEASONS:
    print('Calculating feature importance and SHAP:', season)
    calculate_explanations(season)

In [ ]:
def read_figure_inputs(work_dir):
    predictions, metric_values, importance, shap_data = {}, {}, {}, {}
    for branch, track in [('county_holdout', 'random_county'), ('utility_holdout', 'utility_grouped')]:
        for season in SEASONS:
            folder = Path(work_dir)
            csv_path = folder / f'{branch}_{season}_predictions.csv'
            frame = (pd.read_csv(csv_path, float_precision='round_trip', dtype={'prediction': np.float32})
                     if csv_path.exists() else pd.read_parquet(folder / f'{branch}_{season}_predictions.parquet'))
            predictions[track, season] = frame
            metric_values[track, season] = model_metrics(frame.ELEC_M, frame.prediction)
            if branch == 'county_holdout':
                importance[track, season] = pd.read_csv(
                    folder / f'county_holdout_{season}_feature_importance.csv', float_precision='round_trip').set_index('feature')
                values = pd.read_parquet(folder / f'county_holdout_{season}_shap_values.parquet')[FEATURES]
                explained = pd.read_parquet(folder / f'county_holdout_{season}_explained_500.parquet')[FEATURES]
                shap_data[season] = {'values': values, 'X': explained, 'means': values.abs().mean()}
    return predictions, metric_values, importance, shap_data

FIGURE_NAMES = {'Fig3_random_performance': 'Figure3', 'Fig4_random_feature_importance': 'Figure4', 'Fig5_random_SHAP': 'Figure5', 'utility_performance': 'FigureS1', 'random_residual_distribution': 'FigureS2', 'random_mean_absolute_SHAP': 'FigureS3'}

def save(fig, stem):
    folder = OUTPUT_DIR / 'figures'
    folder.mkdir(parents=True, exist_ok=True)
    for extension in ['png', 'pdf']:
        fig.savefig(folder / f'{FIGURE_NAMES[stem]}.{extension}', format=extension,
                    dpi=600, facecolor='white', bbox_inches='tight')
    plt.close(fig)

In [ ]:
def density_colors(x, y, bins=70):
    xe, ye = (np.linspace(x.min(), x.max(), bins + 1), np.linspace(y.min(), y.max(), bins + 1))
    h, _, _ = np.histogram2d(x, y, bins=[xe, ye])
    xi = np.clip(np.digitize(x, xe) - 1, 0, bins - 1)
    yi = np.clip(np.digitize(y, ye) - 1, 0, bins - 1)
    return np.log1p(h[xi, yi])

def performance(predictions, metrics, track):
    pooled = np.concatenate([predictions[track, s][['ELEC_M', 'prediction']].to_numpy().ravel() for s in SEASONS])
    lo, hi = (500 * np.floor(pooled.min() / 500), 500 * np.ceil(pooled.max() / 500))
    ticks = np.arange(lo, hi + 1, 500)
    fig, axes = plt.subplots(2, 2, figsize=(7.4, 6.67), sharex=True, sharey=True, constrained_layout=True)
    for i, (season, ax) in enumerate(zip(SEASONS, axes.ravel())):
        f, m = (predictions[track, season], metrics[track, season])
        x, y = (f.ELEC_M.to_numpy(), f.prediction.to_numpy())
        ax.scatter(x, y, c=density_colors(x, y), cmap='viridis', s=10, alpha=0.85, linewidths=0)
        ax.plot([lo, hi], [lo, hi], color='black', lw=1, alpha=0.8)
        ax.set(xlim=(lo, hi), ylim=(lo, hi), xticks=ticks, yticks=ticks)
        for spine in ax.spines.values():
            spine.set_linewidth(1.5)
        ax.text(-0.1, 1.1, 'abcd'[i], transform=ax.transAxes, va='top', ha='left', fontsize=16, fontweight='bold', clip_on=False)
        ax.text(0.02, 0.98, f"$R^2$ = {m['r2']:.2f}\nRMSE = {m['rmse']:.2f}\nMAPE = {m['mape_percent']:.2f}%", transform=ax.transAxes, va='top', ha='left', fontsize=12, bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.85, edgecolor='none'))
        ax.tick_params(axis='both', labelsize=11)
        ax.xaxis.set_major_formatter(FormatStrFormatter('%.0f'))
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.0f'))
    fig.supxlabel('Actual Monthly Electricity Consumption (kWh)', fontsize=13)
    fig.supylabel('Predicted Monthly Electricity Consumption (kWh)', fontsize=13)
    save(fig, 'Fig3_random_performance' if track == 'random_county' else 'utility_performance')

def feature_importance(data, labels, track):
    fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)
    for i, (season, ax) in enumerate(zip(SEASONS, axes.ravel())):
        fi = data[track, season]
        total = float(fi.importance.sum())
        chosen = (fi.importance / total * 100).sort_values(ascending=False, kind='stable').head(10).sort_values(kind='stable')
        subset = fi.loc[chosen.index]
        low = subset.p99_low.to_numpy() / total * 100
        high = subset.p99_high.to_numpy() / total * 100
        err = np.vstack([chosen.to_numpy() - low, high - chosen.to_numpy()])
        ax.barh([labels[x] for x in chosen.index], chosen.to_numpy(), xerr=err, color='#0B5C8E', ecolor='black', capsize=3)
        ax.set_xlabel('Relative importance (%)', fontsize=13, fontweight='bold')
        ax.tick_params(axis='both', labelsize=11)
        ax.annotate('abcd'[i], xy=(-0.8, 1.05), xycoords='axes fraction', fontsize=28, fontweight='bold')
        for spine in ax.spines.values():
            spine.set_linewidth(1.3)
    save(fig, 'Fig4_random_feature_importance' if track == 'random_county' else 'utility_feature_importance')

def shap_dots(data, labels):
    fig = plt.figure(figsize=(16, 12))
    gs = matplotlib.gridspec.GridSpec(2, 2, figure=fig, wspace=0.8, hspace=0.32)
    axes = []
    for i, season in enumerate(SEASONS):
        item = data[season]
        ax = fig.add_subplot(gs[i // 2, i % 2])
        axes.append(ax)
        plt.sca(ax)
        np.random.seed(42 + i)
        shap.summary_plot(item['values'].to_numpy().copy(), item['X'].rename(columns=labels), show=False, max_display=10, plot_size=None, color_bar=False)
        ax.set_title('')
        ax.set_xlabel('SHAP value (kWh)', fontsize=13, fontweight='bold')
        if i % 2:
            ax.set_ylabel('')
        ax.tick_params(axis='both', labelsize=11)
        for spine in ax.spines.values():
            spine.set_linewidth(1.3)
    fig.subplots_adjust(left=0.1, right=0.9, top=0.97, bottom=0.08)
    fig.canvas.draw()
    for i, ax in enumerate(axes):
        p = ax.get_position()
        fig.text(p.x0 - 0.23, p.y1 + 0.015, 'abcd'[i], ha='left', va='bottom', fontsize=28, fontweight='bold')
    cmap = LinearSegmentedColormap.from_list('shap_like', ['#1E88E5', '#FF0052'])
    cax = fig.add_axes([0.92, 0.16, 0.008, 0.6])
    sm = ScalarMappable(norm=Normalize(0, 1), cmap=cmap)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cax)
    cb.set_label('Feature value', fontsize=13, fontweight='bold')
    cb.set_ticks([0, 1])
    cb.set_ticklabels(['Low', 'High'])
    cb.ax.tick_params(labelsize=11)
    save(fig, 'Fig5_random_SHAP')

def meanabs(data, labels):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
    for i, (season, ax) in enumerate(zip(SEASONS, axes.ravel())):
        chosen = data[season]['means'].sort_values(ascending=False, kind='stable').head(10).sort_values(kind='stable')
        colors = plt.get_cmap('viridis')(Normalize(vmin=chosen.min(), vmax=chosen.max())(chosen.to_numpy()))
        ax.barh([labels[x] for x in chosen.index], chosen.to_numpy(), color=colors, edgecolor='none')
        ax.set_title(season.title(), fontsize=13, fontweight='bold')
        ax.set_xlabel('Mean |SHAP value| (kWh)', fontsize=12)
        ax.tick_params(axis='both', labelsize=10)
        ax.text(-0.5, 1.05, 'abcd'[i], transform=ax.transAxes, va='top', ha='left', fontsize=25, fontweight='bold')
        for spine in ax.spines.values():
            spine.set_linewidth(1.2)
    save(fig, 'random_mean_absolute_SHAP')

def residual(predictions):
    arrays = {s: predictions['random_county', s].prediction.to_numpy(dtype=float) - predictions['random_county', s].ELEC_M.to_numpy(dtype=float) for s in SEASONS}
    pooled = np.concatenate(list(arrays.values()))
    low, high = np.percentile(pooled, [1, 99])
    limit = max(abs(low), abs(high))
    fig, axes = plt.subplots(2, 2, figsize=(8, 6.8), sharex=True, sharey=True, constrained_layout=True)
    for i, (season, ax) in enumerate(zip(SEASONS, axes.ravel())):
        r = arrays[season]
        ax.hist(r, bins=60, range=(-limit, limit), density=True, color='#f4f2f2', edgecolor='black', linewidth=0.6)
        central = r[(r >= -limit) & (r <= limit)]
        xs = np.linspace(-limit, limit, 500)
        kde = gaussian_kde(central)
        ax.plot(xs, kde(xs), color='black', linewidth=1.5)
        ax.axvline(0, color='black', linestyle='--', linewidth=1)
        ax.text(-0.1, 1.12, 'abcd'[i], transform=ax.transAxes, va='top', ha='left', fontsize=16, fontweight='bold', clip_on=False)
        ax.set_title(season.title(), fontsize=12, fontweight='bold')
        m = {'mean': float(r.mean()), 'median': float(np.median(r)), 'sd': float(r.std(ddof=1))}
        ax.text(0.02, 0.98, f"Mean = {m['mean']:.2f}\nMedian = {m['median']:.2f}\nSD = {m['sd']:.2f}", transform=ax.transAxes, va='top', ha='left', fontsize=10, bbox=dict(boxstyle='round,pad=.25', facecolor='white', alpha=0.85, edgecolor='none'))
        ax.tick_params(axis='both', labelsize=11)
        for spine in ax.spines.values():
            spine.set_linewidth(1.3)
    fig.supxlabel('Residual (kWh)', fontsize=13)
    fig.supylabel('Density', fontsize=13)
    save(fig, 'random_residual_distribution')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FormatStrFormatter
from scipy.stats import gaussian_kde
import shap

plt.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42,
                     'font.family': 'DejaVu Sans', 'font.size': 10})
predictions, scores, importance, shap_data = read_figure_inputs(WORK_DIR)
performance(predictions, scores, 'random_county')
feature_importance(importance, FEATURE_LABELS, 'random_county')
shap_dots(shap_data, FEATURE_LABELS)
performance(predictions, scores, 'utility_grouped')
residual(predictions)
meanabs(shap_data, FEATURE_LABELS)
print('Saved Figures 3–5 and S1–S3 to', OUTPUT_DIR / 'figures')